In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor


# 학습 모델 저장을 위한 라이브러리
import pickle

### 프로젝트 셋팅

In [2]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = 'model/best_model_C,D.dat'
# 교차검증 횟수
cv_count = 10
# 교차 검증
kfold = KFold(n_splits=cv_count, shuffle=True, random_state=1)
# 평가 결과를 담을 리스트
# 필요하다면 다른 것도 만들어주세요
f1_score_list = []
# 학습 모델 이름
model_name_list = []

### 데이터 준비
- 데이터를 읽어오고 필요한 전처리까지 다 한다음 입력데이터는 train_X, 결과데이터는 train_y라는 변수에 담아서 준비해주세요

In [4]:
# 데이터를 읽어온다.
train_df = pd.read_csv('model3_train_CD(5).csv')
test_df = pd.read_csv('model3_test_CD(5).csv')

display(train_df)
display(test_df)

,_1순위카드이용금액,연령,이용금액_R3M_신용체크,이용카드수_신용체크,소지카드수_유효_신용,이용가능여부_해외겸용_본인,수신거부여부_TM,이용금액_일시불_R12M,이용금액대,청구금액_R6M,평잔_일시불_3M
0,3681,40대,196,1,1,0,0,20667,100,88693,1791
1,24493,30대,23988,1,1,1,0,55656,100,165221,6796
2,5933,40대,3904,1,2,1,0,10753,100,127371,772
3,68078,30대,124967,5,3,1,1,360859,100,94241,28376
4,18796,40대,21001,1,1,1,0,34884,100,35723,20399
...,...,...,...,...,...,...,...,...,...,...,...
477943,27337,40대,31187,2,2,1,1,85901,100,74153,4738
477944,35751,50대,42492,1,1,1,0,90222,100,132623,3609
477945,27792,40대,72348,4,3,1,1,169672,100,72474,15212
477946,26357,50대,27636,1,1,1,0,148106,100,99849,9424


,ID,기준년월,청구금액_R6M,이용금액_일시불_R12M,_1순위카드이용금액,이용금액대,연령,이용금액_R3M_신용체크,이용카드수_신용체크,소지카드수_유효_신용,이용가능여부_해외겸용_본인,수신거부여부_TM,평잔_일시불_3M
0,TEST_00000,201807,22151,49063,13852,50,40대,21458,2,2,1,1,3841
1,TEST_00001,201807,32878,7771,11065,50,60대,18681,2,1,1,0,844
2,TEST_00002,201807,71867,73003,27071,100,40대,40758,2,2,0,0,5606
3,TEST_00003,201807,4986,7421,4827,10,40대,5255,1,1,1,0,1510
4,TEST_00004,201807,10758,10493,8011,30,40대,16148,3,1,1,1,1373
...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,TEST_99995,201812,0,0,0,0,60대,0,0,0,0,0,0
599996,TEST_99996,201812,2237,2631,1231,5,30대,3110,1,1,0,0,237
599997,TEST_99997,201812,0,0,0,0,30대,0,0,1,0,0,0
599998,TEST_99998,201812,108420,423882,63592,100,30대,173263,6,3,1,1,17039


In [8]:
# 데이터 프레임을 합친다.
all_df = pd.concat([train_df, test_df])
all_df.reset_index(inplace=True, drop=True)
all_df = all_df.drop(columns=['ID','기준년월'])
all_df

,_1순위카드이용금액,연령,이용금액_R3M_신용체크,이용카드수_신용체크,소지카드수_유효_신용,이용가능여부_해외겸용_본인,수신거부여부_TM,이용금액_일시불_R12M,이용금액대,청구금액_R6M,평잔_일시불_3M
0,3681,40대,196,1,1,0,0,20667,100,88693,1791
1,24493,30대,23988,1,1,1,0,55656,100,165221,6796
2,5933,40대,3904,1,2,1,0,10753,100,127371,772
3,68078,30대,124967,5,3,1,1,360859,100,94241,28376
4,18796,40대,21001,1,1,1,0,34884,100,35723,20399
...,...,...,...,...,...,...,...,...,...,...,...
1077943,0,60대,0,0,0,0,0,0,0,0,0
1077944,1231,30대,3110,1,1,0,0,2631,5,2237,237
1077945,0,30대,0,0,1,0,0,0,0,0,0
1077946,63592,30대,173263,6,3,1,1,423882,100,108420,17039


In [9]:
all_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1077948 entries, 0 to 1077947
Data columns (total 11 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   _1순위카드이용금액      1077948 non-null  int64 
 1   연령              1077948 non-null  object
 2   이용금액_R3M_신용체크   1077948 non-null  int64 
 3   이용카드수_신용체크      1077948 non-null  int64 
 4   소지카드수_유효_신용     1077948 non-null  int64 
 5   이용가능여부_해외겸용_본인  1077948 non-null  int64 
 6   수신거부여부_TM       1077948 non-null  int64 
 7   이용금액_일시불_R12M   1077948 non-null  int64 
 8   이용금액대           1077948 non-null  int64 
 9   청구금액_R6M        1077948 non-null  int64 
 10  평잔_일시불_3M       1077948 non-null  int64 
dtypes: int64(10), object(1)
memory usage: 90.5+ MB


In [10]:
# LabelEncoder 학습
Encoder1 = LabelEncoder()

Encoder1.fit(all_df['연령'])

LabelEncoder()

In [11]:
all_df['연령'] = Encoder1.transform(all_df['연령'])

In [12]:
# Scaler 학습
scalerX = StandardScaler()
scalerX.fit(all_df)

,copy,True
,with_mean,True
,with_std,True


In [13]:
train_df['연령'] = Encoder1.transform(train_df['연령'])

In [14]:
target1=pd.read_parquet(r'data/train/1.회원정보/201807_train_.parquet')
target2=pd.read_parquet(r'data/train/1.회원정보/201808_train_.parquet')
target3=pd.read_parquet(r'data/train/1.회원정보/201809_train_.parquet')
target4=pd.read_parquet(r'data/train/1.회원정보/201810_train_.parquet')
target5=pd.read_parquet(r'data/train/1.회원정보/201811_train_.parquet')
target6=pd.read_parquet(r'data/train/1.회원정보/201812_train_.parquet')

In [15]:
tg_df = pd.concat([
    target1['Segment'],
    target2['Segment'],
    target3['Segment'],
    target4['Segment'],
    target5['Segment'],
    target6['Segment']
])

tg_df = tg_df.reset_index(drop=True).to_frame(name='Segment')
tg_df = tg_df[tg_df['Segment'] != 'E']

tg_df['Segment'] = tg_df['Segment'].apply(
    lambda x: x if x in ['C', 'D'] else 'not CD'
)
tg_df

,Segment
0,D
2,C
3,D
8,C
10,D
...,...
2399979,D
2399987,C
2399993,C
2399996,D


In [16]:
# 라벨 인코더 생성
le = LabelEncoder()

# 문자열 y를 숫자로 변환
tg_df['Segment'] = le.fit_transform(tg_df['Segment'])

In [17]:
# 입력과 결과로 나눈다.
X = train_df
y = tg_df

In [18]:
# 표준화
X2 = scalerX.transform(X)
X2

array([[-7.71967462e-01, -7.57170594e-02, -9.44089683e-01, ...,
         1.30397978e+00,  4.86783492e-01, -5.02247411e-01],
       [ 3.70885632e-01, -9.18749748e-01, -1.69368977e-01, ...,
         1.30397978e+00,  1.64315343e+00, -1.35515106e-03],
       [-6.48302982e-01, -7.57170594e-02, -8.23348914e-01, ...,
         1.30397978e+00,  1.07122417e+00, -6.04227274e-01],
       ...,
       [ 5.52044211e-01, -7.57170594e-02,  1.40534073e+00, ...,
         1.30397978e+00,  2.41707650e-01,  8.40904442e-01],
       [ 4.73243799e-01,  7.67315629e-01, -5.05819423e-02, ...,
         1.30397978e+00,  6.55355296e-01,  2.61650815e-01],
       [-3.11886694e-02, -9.18749748e-01, -1.95451328e-01, ...,
         5.67943352e-02, -2.32774532e-01, -3.81452814e-01]])

In [26]:
scaler_columns = X.columns.tolist()
scaler_columns

['_1순위카드이용금액',
 '연령',
 '이용금액_R3M_신용체크',
 '이용카드수_신용체크',
 '소지카드수_유효_신용',
 '이용가능여부_해외겸용_본인',
 '수신거부여부_TM',
 '이용금액_일시불_R12M',
 '이용금액대',
 '청구금액_R6M',
 '평잔_일시불_3M']

In [19]:
train_X = X2
train_y = y

### 기본 모델 사용하기
- 기본 모델 중에 만족하는 것을 찾았다면 하이퍼 파라미터 튜닝 과정은 생략하세요

In [20]:
model5 = LGBMClassifier(device='cpu', objective='multiclass', num_class=3, verbose=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
r1 = cross_val_score(model5, train_X, train_y, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r1.mean()}')

f1_score_list.append(r1.mean())
model_name_list.append("LGBMClassifier")

평균 f1 Score : 0.7945320398063531


In [21]:
# CPU 기반 XGBoost 모델
model6 = XGBClassifier(
    n_jobs=-1,
    verbosity=0,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

# 교차 검증
kfold = KFold(n_splits=10, shuffle=True, random_state=1)

# f1_weighted 사용
r2 = cross_val_score(model6, train_X, train_y, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r2.mean():.4f}')

f1_score_list.append(r2.mean())
model_name_list.append("XGBClassifier")

평균 f1 Score : 0.7983


In [22]:
df = pd.DataFrame({
    'Model': model_name_list,
    'f1 score': f1_score_list
})
df = df.dropna()

In [23]:
df

,Model,f1 score
0,LGBMClassifier,0.794532
1,XGBClassifier,0.798258


In [24]:
final_model=model6.fit(train_X, train_y)

In [27]:
with open(best_model_path, 'wb') as fp:
    pickle.dump(model6, fp)
    pickle.dump(scalerX, fp)
    pickle.dump(scaler_columns, fp)
    pickle.dump(Encoder1, fp)
    pickle.dump(le, fp)  # ← LabelEncoder 객체 추가

print('저장완료')

저장완료
